In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import hdfstream 
import astropy.units as u
import scipy
import lightcone_io as lc
import unyt
from astropy import constants as const

In [ ]:
#Sets up the connection to the data and reads in the shell array
root_dir = hdfstream.open("cosma", "/")
basedir="FLAMINGO/L1_m9/L1_m9/healpix_maps/nside_4096"
basename="lightcone0"
shell = lc.ShellArray(basedir, basename, remote_dir=root_dir)

In [ ]:
# Index of the shell we're interested in
shell_nr = 0

# Name of the quantity we want to read
map_name = "TotalMass"

# Read the data
map_data = shell[shell_nr][map_name][...]

In [ ]:
# 1. Define the physical constants (using astropy for accuracy)
h = const.h.value       # Planck constant in J*s
k_B = const.k_B.value   # Boltzmann constant in J/K
T_cmb = 2.7255
freq_test = 150.0 # in GHz

x = h*freq_test*1e9/(T_cmb*k_B)
tSZ_freq_dep_func = x*(np.cosh(x/2)/np.sinh(x/2))-4

with hdfstream.open("cosma", "FLAMINGO/L1_m9/L1_m9/integrated_maps/yang26/lightcone0_shells/lensed_tSZ_rot_same_rot.hdf5") as map_file:
  lensed_tSZ_map = map_file["data"][...]
delta_T_tSZ = tSZ_freq_dep_func * lensed_tSZ_map * T_cmb ## in K_CMB

In [ ]:
file_path = 'FLAMINGO/L1_m9/L1_m9/integrated_maps/broxterman24/WL_convergence_Euclid_like_nz_Broxterman24_L1_m9_lc0.hdf5'
with hdfstream.open("cosma", file_path) as data_file:
  kappa_map = data_file["Convergence"][:]

In [ ]:
filename = "FLAMINGO/L1_m9/L1_m9/halo_lightcone/lightcone0/lightcone_halos_0076.hdf5"
halos = lc.HaloLightconeFile(filename, remote_dir=root_dir)
# List of halo properties to read
properties = ("Lightcone/HaloCentre", "BoundSubhalo/TotalMass", "Lightcone/Redshift")

# Read the data
halo_props = halos.read_halos(properties)

In [ ]:
mask = (halo_props["BoundSubhalo/TotalMass"] > 1e5)

In [ ]:
fov_deg = 1
pixel_scale_arcmin = 0.01
npix = int(fov_deg * 60 / pixel_scale_arcmin)

In [ ]:
halo_x = halo_props["Lightcone/HaloCentre"][mask][:, 0]
halo_y = halo_props["Lightcone/HaloCentre"][mask][:, 1]
halo_z = halo_props["Lightcone/HaloCentre"][mask][:, 2] 

ra_deg = np.arctan2(halo_y, halo_x) * 180 / np.pi
dec_deg = np.arcsin(halo_z / np.sqrt(halo_x**2 + halo_y**2 + halo_z**2)) * 180 / np.pi

In [ ]:
tsz_map = lensed_tSZ_map - np.mean(lensed_tSZ_map)

In [ ]:
tsz_list = []
kappa_list = []
for i in range(len(ra_deg)):
    tsz_patch = hp.gnomview(tsz_map, rot=(ra_deg[i], dec_deg[i]), xsize=npix, reso = pixel_scale_arcmin, title = "tSZ", return_projected_map=True, cmap = 'plasma', no_plot=True)
    kappa_patch = hp.gnomview(kappa_map, rot=(ra_deg[i], dec_deg[i]), xsize=npix, reso = pixel_scale_arcmin, title = r"$\kappa$",return_projected_map=True, no_plot=True)
    tsz_list.append(tsz_patch)
    kappa_list.append(kappa_patch)


In [ ]:
tsz_scatter = np.std(tsz_list, axis = 0)
kappa_scatter = np.std(kappa_list, axis = 0)

In [ ]:
def radial_profile(data, center = None, nbins = 20):
    if center is None:
        center = (data.shape[1]//2, data.shape[0]//2)
    y, x = np.indices((data.shape))
    r = np.sqrt((x - center[0])**2 + (y - center[1])**2)
    #r = r.astype(np.int)
    r_flat = r.flatten()
    image_flat = data.flatten()
    
    rmax = np.max(r)
    bins = np.linspace(0, rmax, nbins+1)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    profile_sum, _ = np.histogram(r_flat, bins=bins, weights=image_flat)
    pixel_count, _ = np.histogram(r_flat, bins=bins)
    radial_profile = np.divide(profile_sum, pixel_count, out=np.zeros_like(profile_sum), where=pixel_count!=0)
    
    return bin_centers, radial_profile

In [ ]:
_ , prof_scatter_tsz = radial_profile(tsz_scatter)
_, prof_scatter_kappa = radial_profile(kappa_scatter)
tsz_stack = np.mean(tsz_list, axis = 0)
kappa_stack = np.mean(kappa_list, axis = 0)

In [ ]:
r_tsz, prof_tsz = radial_profile(tsz_stack)
r_kappa, prof_kappa = radial_profile(kappa_stack)

In [ ]:
plt.plot(r_tsz, prof_tsz/np.max(prof_tsz), label = "tSZ")
plt.plot(r_kappa, prof_kappa/np.max(prof_kappa), label = r"$\kappa$") 
#plt.yscale('log') 
plt.legend()

In [ ]:
# Saving data for later use in graphs for comparisons
r_kappa_0_sigma  = r_kappa
r_tsz_0_sigma = r_tsz
prof_tsz_0_sigma = prof_tsz/np.max(prof_tsz)
prof_kappa_0_sigma = prof_kappa/np.max(prof_kappa)


In [ ]:
#Opening the data for the 8sigma case
basedir="FLAMINGO/L1_m9/fgas-8sigma/healpix_maps/nside_4096"
basename="lightcone0"
shell = lc.ShellArray(basedir, basename, remote_dir=root_dir)

file_path = 'FLAMINGO/L1_m9/fgas-8sigma/integrated_maps/broxterman24/WL_convergence_Euclid_like_nz_Broxterman24_fgas-8sigma_lc0.hdf5'
with hdfstream.open("cosma", file_path) as data_file:
  kappa_map = data_file["Convergence"][:]

filename = "FLAMINGO/L1_m9/fgas-8sigma/halo_lightcone/lightcone0/lightcone_halos_0076.hdf5"
halos = lc.HaloLightconeFile(filename, remote_dir=root_dir)

x = h*freq_test*1e9/(T_cmb*k_B)
tSZ_freq_dep_func = x*(np.cosh(x/2)/np.sinh(x/2))-4

with hdfstream.open("cosma", "FLAMINGO/L1_m9/fgas-8sigma/integrated_maps/yang26/lightcone0_shells/lensed_tSZ_rot_same_rot.hdf5") as map_file:
  lensed_tSZ_map = map_file["data"][...]
delta_T_tSZ = tSZ_freq_dep_func * lensed_tSZ_map * T_cmb ## in K_CMB

# List of halo properties to read
properties = ("Lightcone/HaloCentre", "BoundSubhalo/TotalMass", "Lightcone/Redshift")

# Read the data
halo_props = halos.read_halos(properties)

In [ ]:
mask = (halo_props["BoundSubhalo/TotalMass"] > 1e5)
halo_x = halo_props["Lightcone/HaloCentre"][mask][:, 0]
halo_y = halo_props["Lightcone/HaloCentre"][mask][:, 1]
halo_z = halo_props["Lightcone/HaloCentre"][mask][:, 2] 

In [ ]:
tsz_map = lensed_tSZ_map - np.mean(lensed_tSZ_map)
tsz_list = []
kappa_list = []
for i in range(len(ra_deg)):
    tsz_patch = hp.gnomview(tsz_map, rot=(ra_deg[i], dec_deg[i]), xsize=npix, reso = pixel_scale_arcmin, title = "tSZ", return_projected_map=True, cmap = 'plasma', no_plot=True)
    kappa_patch = hp.gnomview(kappa_map, rot=(ra_deg[i], dec_deg[i]), xsize=npix, reso = pixel_scale_arcmin, title = r"$\kappa$",return_projected_map=True, no_plot=True)
    tsz_list.append(tsz_patch)
    kappa_list.append(kappa_patch)

In [ ]:
tsz_scatter = np.std(tsz_list, axis = 0)
kappa_scatter = np.std(kappa_list, axis = 0)
tsz_stack = np.mean(tsz_list, axis = 0)
kappa_stack = np.mean(kappa_list, axis = 0)
_ , prof_scatter_tsz = radial_profile(tsz_scatter)
_, prof_scatter_kappa = radial_profile(kappa_scatter)
r_tsz, prof_tsz = radial_profile(tsz_stack)
r_kappa, prof_kappa = radial_profile(kappa_stack)

In [ ]:
plt.plot(r_tsz * pixel_scale_arcmin, prof_tsz/np.max(prof_tsz), label = "tSZ")
plt.fill_between(r_tsz * pixel_scale_arcmin, (prof_tsz - prof_scatter_tsz)/np.max(prof_tsz-prof_scatter_tsz), (prof_tsz + prof_scatter_tsz)/np.max(prof_tsz+prof_scatter_tsz), alpha=0.3)
plt.plot(r_tsz_0_sigma * pixel_scale_arcmin, prof_tsz_0_sigma, label = "tSZ std", color = 'C0', linestyle = '--')
plt.plot(r_kappa * pixel_scale_arcmin, prof_kappa/np.max(prof_kappa), label = r"$\kappa$") 
plt.plot(r_kappa_0_sigma * pixel_scale_arcmin, prof_kappa_0_sigma, label = r"$\kappa$ std", color = 'C1', linestyle = '--')
#plt.yscale('log') 
plt.legend()

In [ ]:
kappa_mean = np.mean(kappa_stack, axis = 0)
kappa_std = np.std(kappa_stack, axis = 0)
kappa_std_mean = kappa_std / np.sqrt(len(kappa_stack))
prof_kappa_std_mean = prof_kappa_0_sigma / np.sqrt(len(kappa_stack))
prof_tsz_std_mean = prof_tsz_0_sigma / np.sqrt(len(tsz_stack))
tsz_mean = np.mean(tsz_stack, axis = 0)
tsz_std = np.std(tsz_stack, axis = 0)
tsz_std_mean = tsz_std / np.sqrt(len(tsz_stack))


plt.figure(figsize=(10, 6))
plt.plot(r_kappa * pixel_scale_arcmin, prof_kappa / np.max(prof_kappa), label = r"$\kappa$")
plt.fill_between(r_kappa * pixel_scale_arcmin, (prof_kappa + prof_kappa_std_mean) / np.max(prof_kappa), (prof_kappa - prof_kappa_std_mean) / np.max(prof_kappa), alpha=0.3, label = r"$\kappa$ ±1σ Scatter")
plt.legend()
plt.grid(True, alpha=0.3)

In [ ]:
prof_tsz_norm = prof_tsz / np.max(prof_tsz)
prof_kappa_norm = prof_kappa / np.max(prof_kappa)
prof_tsz_0sigma_norm = prof_tsz_0_sigma
prof_kappa_0sigma_norm = prof_kappa_0_sigma

plt.figure(figsize=(10, 6))
plt.plot(r_tsz * pixel_scale_arcmin, prof_tsz_norm, label="tSZ 8σ")
plt.plot(r_tsz_0_sigma * pixel_scale_arcmin, prof_tsz_0sigma_norm, label="tSZ 0σ", color='C0', linestyle='--')
plt.fill_between(
    r_tsz * pixel_scale_arcmin,
    prof_tsz_norm - prof_scatter_tsz / np.max(prof_tsz),
    prof_tsz_norm + prof_scatter_tsz / np.max(prof_tsz),
    alpha=0.3,
    label="tSZ scatter",
)
plt.plot(r_kappa * pixel_scale_arcmin, prof_kappa_norm, label=r"$\kappa$ 8σ")
plt.plot(r_kappa_0_sigma * pixel_scale_arcmin, prof_kappa_0sigma_norm, label=r"$\kappa$ 0σ", color='C1', linestyle='--')
plt.fill_between(
    r_kappa * pixel_scale_arcmin,
    prof_kappa_norm - prof_scatter_kappa / np.max(prof_kappa),
    prof_kappa_norm + prof_scatter_kappa / np.max(prof_kappa),
    alpha=0.3,
    label=r"$\kappa$ scatter",
)
plt.legend()
plt.grid(True, alpha=0.3)